# 02 · Removing the aircraft's own magnetism (Tolles-Lawson)

A magnetometer bolted to an aircraft measures Earth's field **plus** the aircraft's own magnetic field (permanent magnets, induced fields, eddy currents in the airframe). To navigate we must subtract the aircraft part — this is **Tolles-Lawson (T-L) calibration**. It uses a 3-axis fluxgate to model the aircraft field as a function of orientation and removes it.

Two flavours are compared here on a noisy cabin magnetometer (`mag_4_uc`):
- **map-less** — band-pass filters the signal so only the (fast) aircraft field is fitted; needs no map;
- **map-based ("modified")** — fits so the result matches a known anomaly map; more accurate when a map exists.

Following the paper, we fit on the first half of the flight and score on the second half (data the calibration never saw).

In [1]:
import os, sys
while not os.path.isdir('magnavlab') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('repo:', os.getcwd())

repo: /home/wpalka/magnav


In [2]:
import numpy as np
from magnavlab.io import load_flight, load_map, segment_indices
from magnavlab.calibration import BuiltinTL, MapBasedModifiedTL
from magnavlab.geomag import core_field
nav = load_flight('data/Flt1003_train.h5')
mag_map = load_map('data/maps/Eastern_395.h5')
idx = segment_indices(nav, 50713.0, 54497.0)[::5]
lat = np.radians(nav.get('lat')[idx]); lon = np.radians(nav.get('lon')[idx])
# Earth's field (the map-based target) at total-field scale = anomaly map + IGRF core
earth_field = mag_map.value(lat, lon) + core_field(lat, lon, nav.get('alt')[idx])
flux = nav.flux(idx); scalar = nav.get('mag_4_uc')[idx]
half = idx.size // 2                     # paper protocol: 1st half = calibration

## Fit on the 1st half, score on the 2nd half
For each calibrator we print two numbers on the **validation** half: the correlation of the compensated signal with the map (closer to 1 is better) and the residual scatter in nT (smaller is better). The raw, uncompensated signal is shown first for reference.

In [3]:
validation = slice(half, None)
def score(name, calibrator, target=None):
    calibrator.fit(nav.flux(idx[:half]), scalar[:half], nav.dt,
                   target=None if target is None else target[:half])
    comp = calibrator.compensate(flux, scalar, nav.dt)
    corr = np.corrcoef(comp[validation], earth_field[validation])[0, 1]
    print(f'{name:24s} corr with map={corr:.3f}  std(comp-map)={np.std(comp[validation] - earth_field[validation]):.0f} nT')

print('raw mag_4_uc'.ljust(24), f'corr with map={np.corrcoef(scalar[validation], earth_field[validation])[0,1]:.3f}  std={np.std(scalar[validation] - earth_field[validation]):.0f} nT')
score('BuiltinTL (map-less)', BuiltinTL())
score('MapBasedModifiedTL', MapBasedModifiedTL(), target=earth_field)

raw mag_4_uc             corr with map=0.830  std=207 nT
BuiltinTL (map-less)     corr with map=0.878  std(comp-map)=180 nT
MapBasedModifiedTL       corr with map=0.978  std(comp-map)=70 nT


Compensation turns a signal barely related to the map into one that closely matches it. The **map-based modified** variant wins — as in the paper — because it optimises directly against the truth the navigator will use.